In [1]:
pip install camelot-py[cv] opencv-python pdf2image pytesseract pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 107.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.2/313.2 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.7 MB/s eta 0:00:00


PDFInfoNotInstalledError: Unable to get page count. Is poppler installed and in PATH?

In [2]:
!apt-get update
!apt-get install -y poppler-utils


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,302 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/

In [3]:
!pdfinfo -v


pdfinfo version 22.02.0
Copyright 2005-2022 The Poppler Developers - http://poppler.freedesktop.org
Copyright 1996-2011 Glyph & Cog, LLC


In [7]:
import camelot
import pandas as pd
import re
import json
import datetime
import os

import cv2
import numpy as np
from pdf2image import convert_from_path


# -----------------------------------
# Unit pattern dictionary
# -----------------------------------
UNIT_PATTERNS = {
    "tCO2e": r"(tco2e|tonnes? of co2e)",
    "kgCO2e": r"(kgco2e)",
    "GJ": r"\bGJ\b",
    "MJ": r"\bMJ\b",
    "kWh": r"\bkwh\b",
    "MWh": r"\bmwh\b",
    "%": r"\b%\b|percent|percentage",
    "KL": r"\bkl\b",
    "ML": r"\bml\b",
    "m3": r"(m3|cubic meter)",
    "MT": r"\bmt\b",
    "tonnes": r"\btonnes?\b",
    "No": r"(yes|no)"
}


# -----------------------------------
# ESG field mapping
# -----------------------------------
FIELD_METADATA_MAP = {
    "scope_1_emissions_absolute": [r"scope\s*1", r"direct emissions"],
    "scope_2_emissions_lb_mb": [r"scope\s*2", r"indirect emissions"],
    "scope_3_emissions_total": [r"scope\s*3"],
    "renewable_electricity_percent": [r"renewable"],
    "energy_consumption_total": [r"total energy consumption"],
    "water_withdrawal_total": [r"water withdrawal"],
    "waste_generated_total": [r"waste generated"],
    "csr_spend": [r"csr.*expenditure"]
}


# -----------------------------------
# Extract numeric value
# -----------------------------------
def extract_number(text):
    if not text:
        return None
    match = re.search(r"([-+]?\d[\d,]*\.?\d*)", str(text))
    if not match:
        return None
    try:
        val = match.group(1).replace(",", "")
        return float(val) if "." in val else int(val)
    except ValueError:
        return None


# -----------------------------------
# Extract unit
# -----------------------------------
def extract_unit(text):
    if not text:
        return None
    text = str(text).lower()
    for unit, pattern in UNIT_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            return unit
    return None


# -----------------------------------
# Detect scanned page using OpenCV
# -----------------------------------
def is_scanned_page(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    text_pixels = np.sum(edges > 0)
    return text_pixels < 1000


# -----------------------------------
# Main Camelot-based extraction
# -----------------------------------
def extract_brsr_data_camelot(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    results = []

    # --- OpenCV diagnostics (safe block) ---
    scanned_flags = {}
    try:
        images = convert_from_path(pdf_path, dpi=200)
        scanned_flags = {
            i + 1: is_scanned_page(np.array(img))
            for i, img in enumerate(images)
        }
    except Exception as e:
        print("⚠️ OpenCV diagnostics skipped:", e)

    # --- Camelot table extraction ---
    tables = camelot.read_pdf(
        pdf_path,
        pages="all",
        flavor="lattice",
        strip_text="\n"
    )

    if tables.n == 0:
        tables = camelot.read_pdf(
            pdf_path,
            pages="all",
            flavor="stream"
        )

    for table in tables:
        page_num = table.page
        df = table.df.fillna("")

        headers = df.iloc[0].astype(str).str.lower()
        header_unit = None
        for h in headers:
            header_unit = extract_unit(h)
            if header_unit:
                break

        for _, row in df.iterrows():
            row_text = " ".join(map(str, row))

            for key, patterns in FIELD_METADATA_MAP.items():
                if any(re.search(p, row_text, re.IGNORECASE) for p in patterns):

                    value_cell = None
                    unit = None

                    for cell in reversed(row.tolist()):
                        if extract_number(cell) is not None:
                            value_cell = cell
                            unit = extract_unit(cell)
                            break

                    if not unit:
                        unit = header_unit

                    results.append({
                        "company_id": "TATA_MOTORS_IN",
                        "data_point": key,
                        "value": extract_number(value_cell),
                        "unit": unit or "UNKNOWN",
                        "raw_value": str(value_cell).strip(),
                        "page_number": page_num,
                        "year": "2024-25",
                        "extraction_method": "camelot_with_optional_opencv",
                        "scanned_page": scanned_flags.get(page_num),
                        "last_crawled": datetime.date.today().isoformat()
                    })

    return results


# -----------------------------------
# Entry point
# -----------------------------------
if __name__ == "__main__":

    PDF_FILE = r"D:\FITSOL\companyapiusing\Tata_Motors_BRSR_2025.pdf"

    data = extract_brsr_data_camelot(PDF_FILE)

    with open("esg_standardized_output.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Extraction complete. Records: {len(data)}")


FileNotFoundError: PDF not found: D:\FITSOL\companyapiusing\Tata_Motors_BRSR_2025.pdf